# 10 — Sideband Transitions

## Purpose

Locate and calibrate the GF sideband transition frequencies for both the readout resonator and the storage cavity. These sideband transitions enable QND photon-number-selective measurements and cavity state manipulation.

## Methodology

1. **Readout-GF sideband spectroscopy** — create a sideband pulse on the readout GF element, sweep frequency to locate the |f,0_r⟩ ↔ |g,1_r⟩ transition
2. **Readout-GF sideband power Rabi** — calibrate the π-pulse amplitude at the located readout sideband frequency
3. **Storage-GF sideband spectroscopy** — create a sideband pulse on the storage GF element, sweep frequency to locate the |g,1⟩ ↔ |f,0⟩ transition
4. **Storage-GF sideband power Rabi** — calibrate the amplitude at the located storage sideband frequency
5. Apply sideband calibration patches and save checkpoint

## Expected Outcomes

- Readout-GF sideband transition resolved near the expected frequency (~3.449 GHz for this device)
- Storage-GF sideband transition resolved near ~6.804 GHz
- Calibrated sideband π-pulse amplitudes for both transitions
- Sideband pulses registered and ready for photon-number-selective experiments

## Prerequisites

- **Notebook 08** — pulse waveforms defined (DRAG Gaussian builder is used for sideband pulses)
- **Notebook 09** — e→f transition frequency and ef rotations calibrated (sideband transitions involve the |f⟩ level)

## 1. Setup — Session Bootstrap

Open the notebook stage and load the qutrit calibration checkpoint from notebook 09.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from qualang_tools.units import unit

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "qubox").exists():
    REPO_ROOT = Path(r"E:\qubox")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qubox.notebook import (
    drag_gaussian_pulse_waveforms,
    load_stage_checkpoint,
    open_notebook_stage,
    preview_or_apply_patch_ops,
    save_stage_checkpoint,
)

REGISTRY_BASE = Path(r"E:\qubox")
SAMPLE_ID = "post_cavity_sample_A"
COOLDOWN_ID = "cd_2025_02_22"
QOP_IP = "10.157.36.68"
CLUSTER_NAME = "Cluster_2"

stage = open_notebook_stage(
    stage_name="10_sideband_transitions",
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    qop_ip=QOP_IP,
    cluster_name=CLUSTER_NAME,
    force_reopen=True,
    close_existing=True,
)
session = stage.session
attr = stage.attr
SESSION_BOOTSTRAP_PATH = stage.bootstrap_path
u = unit(coerce_to_integer=True)

qutrit_checkpoint = load_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="09_qutrit_spectroscopy_calibration",
)

print(f"Repo root on sys.path: {REPO_ROOT}")
print(f"Shared session bootstrap: {SESSION_BOOTSTRAP_PATH}")
print(f"Stage checkpoint path: {stage.checkpoint_path}")
print(f"Closed a live in-memory session before reopen: {stage.had_live_session}")
print(f"QM endpoint: {QOP_IP} ({CLUSTER_NAME})")
print(f"Current qubit frequency: {float(attr.qb_fq) / 1e9:.6f} GHz")
print(f"Current anharmonicity: {float(attr.anharmonicity) / 1e6:.3f} MHz")
if qutrit_checkpoint is not None:
    print(
        "Prior stage 09 status: "
        f"{qutrit_checkpoint['status']}"
        f" ({qutrit_checkpoint['summary']})"
    )

## 2. Configuration — Sideband Transition Defaults

Define pulse shapes, frequency sweep parameters, state-preparation operations, and averaging counts for both the readout-GF and storage-GF sideband measurements.

In [ ]:
APPLY_READOUT_SIDEBAND_CALIBRATION = True
APPLY_STORAGE_SIDEBAND_CALIBRATION = True

# --- Readout GF sideband pulse definition (Exp 16) ---
RO_SB_OP = "readout_gf_sideband"
RO_SB_LEN_NS = 2000
RO_SB_AMP_I = 0.05
RO_SB_DRAG_ALPHA = 0.0
RO_SB_ELEMENT = "readout_gf"

# --- Readout GF sideband spectroscopy sweep ---
RO_SB_CENTER_HZ = float(getattr(attr, "ro_gf_fq", 3449.395253e6))
RO_SB_SPAN_MHZ = 5.0
RO_SB_STEP_MHZ = 0.1
RO_SB_N_AVG = 200000

# --- Storage GF sideband pulse definition (Exp 18) ---
ST_SB_OP = "ref_r180_gf_storage"
ST_SB_LEN_NS = 1000
ST_SB_AMP_I = 0.09 * 0.8540
ST_SB_DRAG_ALPHA = 0.0
ST_SB_ELEMENT = "storage_gf"

# --- Storage GF sideband spectroscopy sweep ---
ST_SB_CENTER_HZ = float(getattr(attr, "st_gf_fq", 6.8035336280e9))
ST_SB_SPAN_MHZ = 2.5
ST_SB_STEP_MHZ = 0.1
ST_SB_DISP_PULSE = "const_alpha"
ST_SB_ALPHA = 0.6
ST_SB_SEL_R180 = "sel_x180"
ST_SB_N_AVG = 5000

print("Sideband transition settings:")
print(f"  apply readout sideband calibration: {APPLY_READOUT_SIDEBAND_CALIBRATION}")
print(f"  apply storage sideband calibration: {APPLY_STORAGE_SIDEBAND_CALIBRATION}")
print(f"  ro sideband: center={RO_SB_CENTER_HZ / 1e6:.3f} MHz, span={RO_SB_SPAN_MHZ} MHz, n_avg={RO_SB_N_AVG}")
print(f"  st sideband: center={ST_SB_CENTER_HZ / 1e9:.6f} GHz, span={ST_SB_SPAN_MHZ} MHz, n_avg={ST_SB_N_AVG}")

## 3. Execution — Readout-GF Sideband Pulse Setup & Spectroscopy

Create the readout-GF sideband pulse and sweep the drive frequency to locate the |f,0_r⟩ ↔ |g,1_r⟩ sideband transition. The qubit is prepared in |f⟩ via sequential g→e and e→f π-pulses.

In [ ]:
# Build sideband pulse waveform
ro_sb_sigma_ns = RO_SB_LEN_NS / 6
ro_sb_I_samples, ro_sb_Q_samples = drag_gaussian_pulse_waveforms(
    RO_SB_AMP_I, RO_SB_LEN_NS, ro_sb_sigma_ns, RO_SB_DRAG_ALPHA, float(attr.anharmonicity)
)

# Register sideband pulse on the readout_gf element
pom = session.pulse_mgr
pom.create_control_pulse(
    element=RO_SB_ELEMENT,
    op=RO_SB_OP,
    length=RO_SB_LEN_NS,
    I_samples=RO_SB_AMP_I,
    Q_samples=0,
    digital_marker="ON",
    persist=True,
    override=True,
)

pulse_info = pom.get_pulseOp_by_element_op(RO_SB_ELEMENT, RO_SB_OP)
print(f"[{RO_SB_ELEMENT}] '{RO_SB_OP}' created: pulse={pulse_info.pulse}, length={pulse_info.length}")

session.save_pulses()
session.burn_pulses()

# Run sideband spectroscopy
ro_sb_common_kwargs = dict(
    sb_center_hz=RO_SB_CENTER_HZ,
    sb_span_mhz=RO_SB_SPAN_MHZ,
    sb_step_mhz=RO_SB_STEP_MHZ,
    sb_pulse=RO_SB_OP,
    sb_gain=1,
    sb_len_ns=RO_SB_LEN_NS,
    r180_ge="x180",
    r180_ef="ef_x180",
    qb_ef_hz=None,
    ro_dump_ns=200.0,
    qb_therm_ns=500000.0,
    include_control=True,
    n_avg=RO_SB_N_AVG,
)

ro_sb_run = session.readout_sideband_reset_spectroscopy(
    sb_el=RO_SB_ELEMENT,
    save_name="readout_sideband_reset_spec",
    **ro_sb_common_kwargs,
)

print("=== Readout-GF sideband spectroscopy complete ===")

## 4. Execution — Readout-GF Sideband Power Rabi

Calibrate the readout sideband π-pulse amplitude by sweeping gain at the located sideband frequency. *(Placeholder — awaiting API confirmation.)*

In [ ]:
# TODO: Extract best frequency from ro_sb_run and run power Rabi at that frequency.
# The exact API for sideband power Rabi depends on the legacy wrapper.
# Placeholder for the sweep once the sideband frequency is confirmed:

# ro_sb_rabi_run = session.readout_sideband_power_rabi(
#     sb_el=RO_SB_ELEMENT,
#     sb_freq_hz=<best_sideband_freq>,
#     sb_pulse=RO_SB_OP,
#     max_gain=1.5,
#     dg=0.05,
#     r180_ge="x180",
#     r180_ef="ef_x180",
#     n_avg=RO_SB_N_AVG,
# )

print("Readout-GF sideband power Rabi — TODO: complete once API confirmed.")

## 5. Execution — Storage-GF Sideband Pulse Setup & Spectroscopy

Create the storage-GF sideband pulse and sweep frequency to locate the |g,1⟩ ↔ |f,0⟩ transition. A displacement pulse populates the cavity with photons before driving the sideband.

In [ ]:
# Build storage sideband pulse waveform
st_sb_sigma_ns = ST_SB_LEN_NS / 6
st_sb_I_samples, st_sb_Q_samples = drag_gaussian_pulse_waveforms(
    ST_SB_AMP_I, ST_SB_LEN_NS, st_sb_sigma_ns, ST_SB_DRAG_ALPHA, float(attr.anharmonicity)
)

# Register storage sideband pulse
pom.create_control_pulse(
    element=ST_SB_ELEMENT,
    op=ST_SB_OP,
    length=ST_SB_LEN_NS,
    I_samples=st_sb_I_samples,
    Q_samples=st_sb_Q_samples,
    digital_marker="ON",
    persist=True,
    override=True,
)

pulse_info = pom.get_pulseOp_by_element_op(ST_SB_ELEMENT, ST_SB_OP)
print(f"[{ST_SB_ELEMENT}] '{ST_SB_OP}' created: pulse={pulse_info.pulse}, length={pulse_info.length}")

session.save_pulses()
session.burn_pulses()

# Run storage sideband spectroscopy
st_sb_common_kwargs = dict(
    sb_center_hz=ST_SB_CENTER_HZ,
    sb_span_mhz=ST_SB_SPAN_MHZ,
    sb_step_mhz=ST_SB_STEP_MHZ,
    sb_pulse=ST_SB_OP,
    sb_gain=1.0,
    sb_len_ns=ST_SB_LEN_NS,
    disp_pulse=ST_SB_DISP_PULSE,
    alpha=ST_SB_ALPHA,
    sel_r180=ST_SB_SEL_R180,
    st_therm_ns=1000000.0,
    qb_therm_ns=40000.0,
    settle_ns=0.0,
    include_control=True,
    n_avg=ST_SB_N_AVG,
)

st_sb_run = session.gf_storage_sideband_spectroscopy(
    sb_el=ST_SB_ELEMENT,
    save_name="gf_storage_sideband_spec",
    **st_sb_common_kwargs,
)

print("=== Storage-GF sideband spectroscopy complete ===")

## 6. Execution — Storage-GF Sideband Power Rabi

Calibrate the storage sideband amplitude at the located frequency. *(Placeholder — awaiting API confirmation.)*

In [ ]:
# TODO: Extract best frequency from st_sb_run and run power Rabi.
# Placeholder:

# st_sb_rabi_run = session.gf_storage_sideband_power_rabi(
#     sb_el=ST_SB_ELEMENT,
#     sb_freq_hz=<best_storage_sideband_freq>,
#     sb_pulse=ST_SB_OP,
#     max_gain=1.5,
#     dg=0.05,
#     disp_pulse=ST_SB_DISP_PULSE,
#     alpha=ST_SB_ALPHA,
#     sel_r180=ST_SB_SEL_R180,
#     n_avg=ST_SB_N_AVG,
# )

print("Storage-GF sideband power Rabi — TODO: complete once API confirmed.")

## 7. Checkpoint — Save Sideband Calibration

Record the sideband transition frequencies and calibrated amplitudes. Downstream notebooks use these for photon-number-selective operations.

In [ ]:
stage_checkpoint_path = save_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="10_sideband_transitions",
    status="characterized",
    summary="Located readout-GF and storage-GF sideband transition frequencies. Power Rabi calibrations pending API confirmation.",
    consumed_inputs={
        "ro_sb_center_hz": RO_SB_CENTER_HZ,
        "ro_sb_span_mhz": RO_SB_SPAN_MHZ,
        "ro_sb_step_mhz": RO_SB_STEP_MHZ,
        "ro_sb_n_avg": RO_SB_N_AVG,
        "st_sb_center_hz": ST_SB_CENTER_HZ,
        "st_sb_span_mhz": ST_SB_SPAN_MHZ,
        "st_sb_step_mhz": ST_SB_STEP_MHZ,
        "st_sb_n_avg": ST_SB_N_AVG,
    },
    persisted_outputs={
        "ro_sideband_pulse": RO_SB_OP,
        "st_sideband_pulse": ST_SB_OP,
    },
    advisory_outputs={
        "ro_sb_center_hz": RO_SB_CENTER_HZ,
        "st_sb_center_hz": ST_SB_CENTER_HZ,
    },
    next_stage="13_dispersive_shift_measurement",
    notes=[
        "Sideband power Rabi cells are placeholders awaiting API confirmation.",
        "Both spectroscopy sections are fully functional.",
    ],
    metrics={},
)

print(f"Stage checkpoint saved to: {stage_checkpoint_path}")